In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

In [4]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
BATCH_SIZE = 64
LEARNING_RATE = 0.001
EPOCHS = 10

In [5]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

In [6]:
train_dataset = datasets.MNIST(root = './data', train = True, transform = transform, download = True)
test_dataset = datasets.MNIST(root = './data', train = False, transform = transform, download = True)

In [7]:
train_loader = DataLoader(dataset = train_dataset, batch_size= BATCH_SIZE, shuffle = True)
test_loader = DataLoader(dataset = test_dataset, batch_size= BATCH_SIZE, shuffle = False)

In [12]:
class ConvolutionalNeuralNetwork(nn.Module):
    def __init__(self, num_classes = 10):
        super(ConvolutionalNeuralNetwork, self).__init__()

        self.features = nn.Sequential(
            nn.Conv2d(in_channels= 1, out_channels = 16, kernel_size = 3, stride = 1, padding = 1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size= 2, stride = 2),

            nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, stride = 1, padding = 1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size = 2, stride = 2)
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 7 * 7, 128),
            nn.ReLU(),
            nn.Linear(128, num_classes)
        )
    def forward(self, x):
            x = self.features(x)
            x = self.classifier(x)
            return x

model = ConvolutionalNeuralNetwork(num_classes = 10).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr = LEARNING_RATE)

In [13]:
def train_model():
    model.train()

    for epoch in range(EPOCHS):
        running_loss = 0.0
        for batch_idx, (data, targets) in enumerate(train_loader):
            data, targets = data.to(device), targets.to(device)

            outputs = model(data)
            loss = criterion(outputs, targets)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            running_loss += loss.item()

        print(f"Epoch [{epoch + 1}/{EPOCHS}], Loss: {running_loss/len(train_loader):.4f}")


In [14]:
def evaluate_model():
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for data, targets in test_loader:
            data, targets = data.to(device), targets.to(device)
            outputs = model(data)
            _, predicted = torch.max(outputs.data, 1)
            total += targets.size(0)
            correct += (predicted == targets).sum().item()

    print(f"Accuracy on test set: {100 * correct / total:.2f}%")

if __name__ == '__main__':
    train_model()
    evaluate_model()

Epoch [1/10], Loss: 0.2015
Epoch [2/10], Loss: 0.0568
Epoch [3/10], Loss: 0.0401
Epoch [4/10], Loss: 0.0296
Epoch [5/10], Loss: 0.0234
Epoch [6/10], Loss: 0.0174
Epoch [7/10], Loss: 0.0152
Epoch [8/10], Loss: 0.0117
Epoch [9/10], Loss: 0.0096
Epoch [10/10], Loss: 0.0079
Accuracy on test set: 99.01%
